In [40]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

In [41]:
all_data = np.loadtxt('diabetes.csv.gz',delimiter=',',dtype=np.float32)
all_data

array([[-0.294118 ,  0.487437 ,  0.180328 , ..., -0.53117  , -0.0333333,
         0.       ],
       [-0.882353 , -0.145729 ,  0.0819672, ..., -0.766866 , -0.666667 ,
         1.       ],
       [-0.0588235,  0.839196 ,  0.0491803, ..., -0.492741 , -0.633333 ,
         0.       ],
       ...,
       [-0.411765 ,  0.21608  ,  0.180328 , ..., -0.857387 , -0.7      ,
         1.       ],
       [-0.882353 ,  0.266332 , -0.0163934, ..., -0.768574 , -0.133333 ,
         0.       ],
       [-0.882353 , -0.0653266,  0.147541 , ..., -0.797609 , -0.933333 ,
         1.       ]], shape=(759, 9), dtype=float32)

In [42]:
train_x = all_data[:,:-1]
train_y = all_data[:,[-1]]
train_y.shape

(759, 1)

In [43]:
class Dia_Data(Dataset):
    def __init__(self, train_x, train_y):
        self.x_data = torch.from_numpy(train_x).to('cuda')
        self.y_data = torch.from_numpy(train_y).to('cuda')
        self.len = train_y.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

dia_data = Dia_Data(train_x, train_y)
train_data = DataLoader(dataset=dia_data, batch_size=32, shuffle=True, num_workers=0)

In [44]:
class NeuralNetWork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = torch.nn.Linear(8,4,bias=True)
        self.linear2 = torch.nn.Linear(4,2,bias=False)
        self.linear3 = torch.nn.Linear(2,1,bias=False)
        self.hid_act = torch.nn.ReLU()
        self.ans_act = torch.nn.Sigmoid()

    def forward(self,x):
        x = self.hid_act(self.linear1(x))
        x = self.hid_act(self.linear2(x))
        return self.ans_act(self.linear3(x))

model = NeuralNetWork().to('cuda')
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-4)

In [45]:
for epoch in range(1, 1001):
    for i, data in enumerate(train_data, 0):
        x, y = data
        y_hat = model(x)
        loss = criterion(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"epoch:{epoch};i:{i};loss:{loss.item()}")

epoch:1;i:0;loss:0.6917990446090698
epoch:1;i:1;loss:0.6892932057380676
epoch:1;i:2;loss:0.6895327568054199
epoch:1;i:3;loss:0.6882205009460449
epoch:1;i:4;loss:0.6880311369895935
epoch:1;i:5;loss:0.6871561408042908
epoch:1;i:6;loss:0.6785274744033813
epoch:1;i:7;loss:0.6754481792449951
epoch:1;i:8;loss:0.6713603138923645
epoch:1;i:9;loss:0.6822566986083984
epoch:1;i:10;loss:0.6669425964355469
epoch:1;i:11;loss:0.6614360213279724
epoch:1;i:12;loss:0.6709660291671753
epoch:1;i:13;loss:0.6868686676025391
epoch:1;i:14;loss:0.6564106941223145
epoch:1;i:15;loss:0.6612293720245361
epoch:1;i:16;loss:0.6544097065925598
epoch:1;i:17;loss:0.6680464744567871
epoch:1;i:18;loss:0.6504063010215759
epoch:1;i:19;loss:0.6487037539482117
epoch:1;i:20;loss:0.6239107847213745
epoch:1;i:21;loss:0.6428712606430054
epoch:1;i:22;loss:0.6705333590507507
epoch:1;i:23;loss:0.5994458794593811
epoch:2;i:0;loss:0.6071650981903076
epoch:2;i:1;loss:0.621624767780304
epoch:2;i:2;loss:0.5503380298614502
epoch:2;i:3;los